# Brazil Occupation Pay Gap

Folha de S.Paulo published a story on 20 July 2026 about the highest-paid occupations in Brazil. This notebook uses that story as a starting point, then slows the ranking down.

The question is not only who ranks first. The better question is whether the average represents the middle of the occupation, or whether a smaller high-income tail is pulling the number up.

The analysis uses PNAD Continua/IBGE microdata from Q1 2026. It is descriptive work. No causal claim is made.


## How To Reproduce

Run this from the project root. Raw PNAD files are cached outside the repo by default.

```powershell
Rscript requirements.R
Rscript R/run_analysis.R
Rscript R/validate_outputs.R
```


In [1]:
library(readr)
library(jsonlite)
library(dplyr)

summary <- fromJSON("../data/processed/analysis_summary.json")
cat(sprintf("Run generated at: %s\n", summary$generated_at))
cat(sprintf("Rows imported: %s\n", format(summary$rows_imported, big.mark = ",")))
cat(sprintf("Analytic rows: %s\n", format(summary$rows_analytic, big.mark = ",")))
cat(sprintf("Occupations found: %s\n", summary$occupations_total))
cat(sprintf("Occupations after reliability filter: %s\n", summary$occupations_after_filter))
cat(sprintf("Minimum occupation sample: %s\n", summary$min_sample))
cat(sprintf("Minimum occupation-sex cell sample: %s\n", summary$min_cell_sample))


Run generated at: 2026-07-20 16:02:45 -03
Rows imported: 511,149
Analytic rows: 223,762
Occupations found: 430
Occupations after reliability filter: 266
Minimum occupation sample: 80
Minimum occupation-sex cell sample: 40


## Ranking Result

The first result already changes the story a bit. `Diretores gerais e gerentes gerais` lead by mean income, but their median is far lower than the mean. That points to a stretched distribution inside the occupation.

Among `Medicos especialistas`, the mean is lower than the directors group, but the median is higher. That is a different kind of high-income occupation: less about a huge top tail, more about a high middle.

| Rank | Occupation | Mean income | Weighted median | Sample n |
| --- | --- | --- | --- | --- |
| 1 | 1120 - Diretores gerais e gerentes gerais | R$ 22.416 | R$ 11.000 | 219 |
| 2 | 2212 - Médicos especialistas | R$ 19.293 | R$ 15.000 | 837 |
| 3 | 3355 - Inspetores de polícia e detetives | R$ 14.252 | R$ 11.000 | 176 |
| 4 | 0411 - Oficiais de polícia militar | R$ 13.924 | R$ 12.300 | 91 |
| 5 | 2619 - Profissionais em direito não classificados anteriormente | R$ 13.704 | R$ 12.000 | 447 |
| 6 | 1211 - Dirigentes financeiros | R$ 13.693 | R$ 8.000 | 246 |
| 7 | 1330 - Dirigentes de serviços de tecnologia da informação e comunicações | R$ 13.570 | R$ 10.000 | 250 |
| 8 | 1349 - Dirigentes e gerentes de serviços profissionais não classificados anteriormente | R$ 12.840 | R$ 8.000 | 127 |
| 9 | 2211 - Médicos gerais | R$ 12.551 | R$ 12.000 | 544 |
| 10 | 1323 - Dirigentes de empresas de construção | R$ 12.483 | R$ 10.000 | 133 |


![Top occupations by income](../figures/top20_occupations_income.png)


In [2]:
occupation_rank <- read_csv("../data/processed/occupation_rank_q1_2026.csv", show_col_types = FALSE)
occupation_rank |>
  select(rank, occupation_label, avg_income, median_income, sample_n) |>
  slice_head(n = 10)


| Rank | Occupation | Mean income | Weighted median | Sample n |
| --- | --- | --- | --- | --- |
| 1 | 1120 - Diretores gerais e gerentes gerais | R$ 22.416 | R$ 11.000 | 219 |
| 2 | 2212 - Médicos especialistas | R$ 19.293 | R$ 15.000 | 837 |
| 3 | 3355 - Inspetores de polícia e detetives | R$ 14.252 | R$ 11.000 | 176 |
| 4 | 0411 - Oficiais de polícia militar | R$ 13.924 | R$ 12.300 | 91 |
| 5 | 2619 - Profissionais em direito não classificados anteriormente | R$ 13.704 | R$ 12.000 | 447 |
| 6 | 1211 - Dirigentes financeiros | R$ 13.693 | R$ 8.000 | 246 |
| 7 | 1330 - Dirigentes de serviços de tecnologia da informação e comunicações | R$ 13.570 | R$ 10.000 | 250 |
| 8 | 1349 - Dirigentes e gerentes de serviços profissionais não classificados anteriormente | R$ 12.840 | R$ 8.000 | 127 |
| 9 | 2211 - Médicos gerais | R$ 12.551 | R$ 12.000 | 544 |
| 10 | 1323 - Dirigentes de empresas de construção | R$ 12.483 | R$ 10.000 | 133 |

## Composition Inside The Ranking

The top of the income ranking is not only about occupation names. It also reflects who is concentrated inside each occupation. The chart below shows the estimated gender composition among the 15 occupations with highest mean income.

![Gender composition](../figures/top15_gender_composition.png)


## Gender Gap Sensitivity

The table below only includes occupation-sex cells with enough unweighted observations. These are descriptive differences in survey-weighted mean income; they should not be read as causal evidence.

| Occupation | Men mean | Women mean | Women vs men |
| --- | --- | --- | --- |
| 1120 - Diretores gerais e gerentes gerais | R$ 26.712 | R$ 12.656 | -52,6% |
| 1211 - Dirigentes financeiros | R$ 17.345 | R$ 9.970 | -42,5% |
| 1221 - Dirigentes de vendas e comercialização | R$ 13.490 | R$ 9.456 | -29,9% |
| 1346 - Gerentes de sucursais de bancos, de serviços financeiros e de seguros | R$ 13.268 | R$ 9.528 | -28,2% |
| 1330 - Dirigentes de serviços de tecnologia da informação e comunicações | R$ 14.383 | R$ 10.459 | -27,3% |


In [3]:
gender_gap <- read_csv("../data/processed/top15_gender_gap.csv", show_col_types = FALSE)
gender_gap |>
  select(occupation_label, avg_income_Homem, avg_income_Mulher, gap_women_vs_men) |>
  slice_head(n = 5)


| Occupation | Men mean | Women mean | Women vs men |
| --- | --- | --- | --- |
| 1120 - Diretores gerais e gerentes gerais | R$ 26.712 | R$ 12.656 | -52,6% |
| 1211 - Dirigentes financeiros | R$ 17.345 | R$ 9.970 | -42,5% |
| 1221 - Dirigentes de vendas e comercialização | R$ 13.490 | R$ 9.456 | -29,9% |
| 1346 - Gerentes de sucursais de bancos, de serviços financeiros e de seguros | R$ 13.268 | R$ 9.528 | -28,2% |
| 1330 - Dirigentes de serviços de tecnologia da informação e comunicações | R$ 14.383 | R$ 10.459 | -27,3% |

## Less Obvious Findings

- The top rank is fragile. Directors have the highest mean, but also one of the widest confidence intervals in the top group.
- Legal occupations show long upper tails. Some labels have means close to twice their medians.
- Representation and pay do not move together neatly. Some high-income occupations have many women, but still show a large descriptive income gap by sex.
- Management is not one clean category. Financial directors, sales directors and general directors have different income distributions and gender compositions.

The short version: a wage ranking is not a ladder. It is a map of uneven distributions.


## Interpretation

This is the point I would keep from the notebook: the same average can hide very different labor-market stories. One occupation may have a high middle. Another may have a smaller group at the top pulling the mean upward.

The gender and race/color estimates are descriptive gaps, not proof of discrimination. PNAD does not observe the exact employer, role, seniority, contract details or productivity measures needed for that claim.

For the full method and references, see `docs/methodology.md`, `docs/insights.md` and `docs/literature-review.md`.
